In [1]:
import os
import re
import time

import nltk
import spacy
import openpyxl

from openai import OpenAI

from rdflib import Graph, URIRef
from rdflib.namespace import OWL, RDF, RDFS

from olaf.pipeline.pipeline_schema import Pipeline
from olaf.repository.corpus_loader.text_corpus_loader import TextCorpusLoader
from olaf.pipeline.data_preprocessing.token_selector_data_preprocessing import TokenSelectorDataPreprocessing
from olaf.commons.spacy_processing_tools import is_not_stopword, is_not_punct
from olaf.commons.llm_tools import LLMGenerator
from olaf.pipeline.pipeline_component.term_extraction.llm_term_extraction import LLMTermExtraction
from olaf.pipeline.pipeline_component.candidate_term_enrichment.llm_based_enrichment import LLMBasedTermEnrichment
from olaf.pipeline.pipeline_component.concept_relation_extraction.llm_based_concept_extraction import LLMBasedConceptExtraction
from olaf.pipeline.pipeline_component.concept_relation_extraction.llm_based_relation_extraction import LLMBasedRelationExtraction
from olaf.pipeline.pipeline_component.axiom_extraction.llm_based_axiom_extraction import LLMBasedOWLAxiomExtraction
from olaf.commons.prompts import (
    openai_prompt_concept_term_extraction,
    openai_prompt_relation_term_extraction,
    openai_prompt_term_enrichment,
    openai_prompt_concept_extraction,
    openai_prompt_relation_extraction,
    openai_prompt_owl_axiom_extraction,
)
from olaf.data_container import Concept, KnowledgeRepresentation, Relation
from olaf.repository.serialiser.rdf_owl_serialisers.base_owl_serialiser import BaseOWLSerialiser

In [2]:
os.environ["JAVA_EXE"]  = "/usr/bin/java"
os.environ["ROBOT_JAR"] = "/home/selassri/tools/robot/robot.jar"
os.environ["DATA_PATH"] = "/home/selassri/Documents/OLAF_DEMO/output"
os.makedirs(os.environ["DATA_PATH"], exist_ok=True)

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

nlp = spacy.load("en_core_web_lg")

In [3]:
os.environ["OPENAI_API_KEY"]  = "dummy"
os.environ["OPENAI_BASE_URL"] = "http://localhost:8000/v1"
_MODEL = "openai/gpt-oss-20b"


def _clean_output(text: str) -> str:
    text = re.sub(r"```\w*", "", text).strip()
    text = (text.replace("“", '"').replace("”", '"')
                .replace("‘", "'").replace("’", "'"))

    if text.lstrip().startswith(("@prefix", "@base", "<http")):
        return text

    brace_pos   = text.find('{')
    bracket_pos = text.find('[')

    if bracket_pos != -1 and (brace_pos == -1 or bracket_pos < brace_pos):
        order = [('[', ']'), ('{', '}')]
    else:
        order = [('{', '}'), ('[', ']')]

    for open_ch, close_ch in order:
        start = text.find(open_ch)
        if start == -1:
            continue
        depth = 0
        for i, ch in enumerate(text[start:], start):
            depth += (ch == open_ch) - (ch == close_ch)
            if depth == 0:
                return text[start:i + 1]
    return text


class LocalLLMGenerator(LLMGenerator):
    """vLLM-backed generator with retry/backoff for transient failures."""

    def check_resources(self):
        pass

    def generate_text(self, prompt):
        msgs = ([{"role": "user", "content": prompt}]
                if isinstance(prompt, str) else prompt)
        last_exc = None
        for attempt in range(4):
            try:
                raw = (OpenAI()
                       .chat.completions
                       .create(model=_MODEL, messages=msgs)
                       .choices[0].message.content or "")
                cleaned = _clean_output(raw)
                if cleaned:
                    return cleaned
                raise ValueError("Empty response from model")
            except Exception as exc:
                last_exc = exc
                wait = 2 ** attempt
                print(f"  [retry {attempt+1}/4 after {wait}s] {exc}")
                time.sleep(wait)
        raise RuntimeError(f"LLM call failed after 4 attempts: {last_exc}")

In [4]:
def owl_to_seed_kr(owl_path: str) -> KnowledgeRepresentation:
    g = Graph()
    g.parse(owl_path)

    def get_label(uri):
        lbl = g.value(uri, RDFS.label)
        return str(lbl) if lbl else str(uri).rsplit("#", 1)[-1].rsplit("/", 1)[-1]

    concepts = set()
    for cls in g.subjects(RDF.type, OWL.Class):
        if isinstance(cls, URIRef):
            concepts.add(Concept(label=get_label(cls)))

    relations = set()
    for prop in g.subjects(RDF.type, OWL.ObjectProperty):
        if isinstance(prop, URIRef):
            relations.add(Relation(label=get_label(prop)))

    return KnowledgeRepresentation(concepts=concepts, relations=relations)


seed_kr = owl_to_seed_kr("./data/ontologie/ContextOntology-COInd4.owl")
print(f"Seed ontology loaded: {len(seed_kr.concepts)} concepts, {len(seed_kr.relations)} relations")

Seed ontology loaded: 47 concepts, 51 relations


In [5]:
llm = LocalLLMGenerator()

pipeline = Pipeline(
    spacy_model=nlp,
    corpus_loader=TextCorpusLoader(corpus_path="./data/XQuality/alarm.txt"),
    seed_kr=seed_kr,
    preprocessing_components=[
        TokenSelectorDataPreprocessing(
            selector=lambda t: is_not_stopword(t) and is_not_punct(t)
        )
    ],
    pipeline_components=[],
)
pipeline.run()
print(f"Corpus loaded: {len(pipeline.corpus)} document(s)")

[2026-07-01 13:58:48,394] [WARNING] [token_selector_data_preprocessing] [__init__] [Data preprocessing token sequence attribute not set by the user. 
                By default the token sequence attribute selected_tokens will be used.]


Corpus loaded: 1014 document(s)


In [6]:
# Phase 1 — extract concept candidate terms
LLMTermExtraction(
    prompt_template=openai_prompt_concept_term_extraction,
    llm_generator=llm,
).run(pipeline)
print(f"Candidate terms: {len(pipeline.candidate_terms)}")

Candidate terms: 584


In [7]:
# Phase 2 — enrich candidate terms
LLMBasedTermEnrichment(
    prompt_template=openai_prompt_term_enrichment,
    llm_generator=llm,
).run(pipeline)
print(f"Candidate terms after enrichment: {len(pipeline.candidate_terms)}")

[2026-07-01 14:45:06,428] [ERROR] [llm_based_enrichment] [_enrich_cterm] [LLM generator output is not in the expected format. The candidate term input X3.5 can not be enriched.]
[2026-07-01 14:46:30,933] [ERROR] [llm_based_enrichment] [_enrich_cterm] [LLM generator output is not in the expected format. The candidate term axis feed stop can not be enriched.]
[2026-07-01 14:52:59,976] [ERROR] [llm_based_enrichment] [_enrich_cterm] [LLM generator output is not in the expected format. The candidate term X can not be enriched.]
[2026-07-01 15:07:28,776] [ERROR] [llm_based_enrichment] [_enrich_cterm] [LLM generator output is not in the expected format. The candidate term user page can not be enriched.]
[2026-07-01 15:12:50,403] [ERROR] [llm_based_enrichment] [_enrich_cterm] [LLM generator output is not in the expected format. The candidate term Alarm No. 3028 can not be enriched.]


Candidate terms after enrichment: 584


In [8]:
# Phase 3 — group terms into concepts
LLMBasedConceptExtraction(
    prompt_template=openai_prompt_concept_extraction,
    llm_generator=llm,
).run(pipeline)
print(f"Concepts: {len(pipeline.kr.concepts)}")

Concepts: 60


In [9]:
# Phase 4 — extract relation candidate terms
LLMTermExtraction(
    prompt_template=openai_prompt_relation_term_extraction,
    llm_generator=llm,
).run(pipeline)
print(f"Candidate terms (relations): {len(pipeline.candidate_terms)}")

  [retry 1/4 after 1s] Empty response from model
Candidate terms (relations): 326


In [10]:
# Phase 5 — extract relations between concepts
LLMBasedRelationExtraction(
    prompt_template=openai_prompt_relation_extraction,
    llm_generator=llm,
    concept_max_distance=11,
).run(pipeline)
print(f"Relations: {len(pipeline.kr.relations)}")

Relations: 82


In [11]:
# Phase 6 — generate OWL axioms via LLM
LLMBasedOWLAxiomExtraction(
    prompt_template=openai_prompt_owl_axiom_extraction,
    llm_generator=llm,
    namespace="http://semanticweb.org/STEaMINg/ContextOntology-COInd4#",
).run(pipeline)
print(f"Concepts  : {len(pipeline.kr.concepts)}")
print(f"Relations : {len(pipeline.kr.relations)}")

Concepts  : 60
Relations : 82


In [12]:
# Export triplets -> Excel
os.makedirs("output", exist_ok=True)

wb_out = openpyxl.Workbook()
ws_out = wb_out.active
ws_out.append(["Source Node", "Target Node", "Relation"])
for r in sorted(
    pipeline.kr.relations,
    key=lambda x: (
        x.source_concept.label if x.source_concept else "?",
        x.destination_concept.label if x.destination_concept else "?",
    )
):
    src = r.source_concept.label if r.source_concept else "?"
    dst = r.destination_concept.label if r.destination_concept else "?"
    ws_out.append([src, dst, r.label])
wb_out.save("output/llm_alarm_triplets.xlsx")
print(f"Exported {len(pipeline.kr.relations)} triplets -> output/llm_alarm_triplets.xlsx")

Exported 82 triplets -> output/llm_alarm_triplets.xlsx


In [13]:
# Export ontology -> TTL (Turtle) using OLAF's BaseOWLSerialiser
owl_serialiser = BaseOWLSerialiser(
    base_uri="http://semanticweb.org/STEaMINg/ContextOntology-COInd4#",
    keep_all_labels=True,
)
owl_serialiser.build_graph(pipeline.kr)
owl_serialiser.export_graph(
    file_path="output/llm_alarm_ontology.ttl",
    rdf_format="turtle",
)
print("Exported ontology -> output/llm_alarm_ontology.ttl")

Exported ontology -> output/llm_alarm_ontology.ttl
